# Import and Setup

In [20]:
import os
import json
import ast
from openai import OpenAI

openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key
client = OpenAI()
MODEL = "gpt-5.4"
REASONING_EFFORT = "medium"  # "low", "medium", "high", or None to disable

# LLM Functions

In [21]:
REQUIREMENTS = """- Must run quickly on a MacBook with 36GB RAM (Apple Silicon); use device='mps' where supported
- Use a single small transformer-based model (e.g. distilbert, all-MiniLM-L6-v2, or similarly lightweight models via transformers.pipeline or sentence-transformers)
- No training, fine-tuning, or weight updates — load a pretrained model and evaluate it directly (zero-shot or task-specific pretrained checkpoint)
- No hyperparameter tuning or loops over multiple models/configurations — pick one and run it
- Only one model and one dataset/subset
- Only code cells (no markdown cells)
- No plots or visualizations"""

In [22]:
def generate_data_science_tasks(n: int = 10) -> list:
    prompt = f"""Brainstorm a list of {n} descriptions of AI tasks that can be evaluated using a modern AI model and HuggingFace datasets.

Requirements for each task:
{REQUIREMENTS}
- Restrict to tasks with datasets that have less than a million samples
- Each description should specify both the task type and the dataset

Return ONLY a valid Python list of strings, no explanation."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    return ast.literal_eval(raw)


# tasks = generate_data_science_tasks(10)
# tasks

In [23]:
def _call(messages: list, **kwargs) -> str:
    """Make a model call, optionally with reasoning_effort."""
    extra = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT is not None else {}
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **extra,
        **kwargs,
    )
    return response.choices[0].message.content


def _parse_notebook_raw(raw: str) -> dict:
    """Strip markdown fences and parse notebook JSON."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.rsplit("```", 1)[0].strip()
    return json.loads(raw)


def generate_notebook(task: str, notebook_dir: str) -> str:
    os.makedirs(notebook_dir, exist_ok=True)
    log_path = os.path.join(notebook_dir, "generation.log")

    def log(msg: str):
        with open(log_path, "a") as lf:
            lf.write(msg + "\n")

    # Step 0: Name the notebook
    notebook_name = _call([{"role": "user", "content": f"""Generate a short, descriptive filename for a Jupyter notebook about this AI task:

Task: {task}

Requirements:
- Use snake_case
- End with .ipynb
- Be concise (3-6 words)
- Return ONLY the filename, nothing else."""}]).strip()
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"
    log(f"=== [generate_notebook] Step 0: Name ===\n{notebook_name}\n")

    # Step 1: Plan
    plan = _call([{"role": "user", "content": f"""You are an expert data scientist. Plan a Jupyter notebook workflow for this AI task:

Task: {task}

Requirements:
{REQUIREMENTS}

Write a concise step-by-step plan for the notebook that respects all requirements above."""}]).strip()
    log(f"=== [generate_notebook] Step 1: Plan ===\n{plan}\n")

    # Step 2: Generate notebook JSON
    raw = _call([{"role": "user", "content": f"""You are an expert data scientist. Generate a complete Jupyter notebook as valid JSON for this AI task.

Task: {task}

Plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}])
    log(f"=== [generate_notebook] Step 2: Raw notebook JSON (first 500 chars) ===\n{raw[:500]}\n")

    nb = _parse_notebook_raw(raw)
    path = os.path.join(notebook_dir, notebook_name)
    with open(path, "w") as f:
        json.dump(nb, f, indent=1)
    log(f"Notebook saved to {path}")
    return path

In [24]:
def generate_n_variations(notebook_path: str, n: int = 3, overwrite=False) -> list:
    with open(notebook_path, "r") as f:
        original_nb = json.load(f)

    cells_text = []
    for cell in original_nb.get("cells", []):
        source = "".join(cell.get("source", []))
        if source.strip():
            cells_text.append(source)
    original_content = "\n\n---\n\n".join(cells_text)

    original_name = os.path.splitext(os.path.basename(notebook_path))[0]
    notebook_dir = os.path.dirname(notebook_path)
    variations_dir = os.path.join(notebook_dir, f"variations_{original_name}")
    if os.path.exists(variations_dir) and not overwrite:
        return []
    os.makedirs(variations_dir, exist_ok=True)

    log_path = os.path.join(variations_dir, "generation.log")

    def log(msg: str):
        with open(log_path, "a") as lf:
            lf.write(msg + "\n")

    # Step 0: Plan all variations upfront as methodologically distinct pipelines
    raw_plans = _call([{"role": "user", "content": f"""You are an expert ML engineer. Plan {n} variations of the following Jupyter notebook, where each variation tests a genuinely different ML pipeline methodology for the same task.

Original notebook:
{original_content}

Your goal is to explore the space of correct machine learning pipeline methodologies for this task — not to produce cosmetic variations of the same approach.

Requirements that every variation must satisfy:
{REQUIREMENTS}
- Stay on the same task type as the original notebook

For each variation, write:
- A snake_case filename (without .ipynb)
- A methodology summary: one sentence explicitly stating the inference entry point, representation strategy, and scoring approach — and how it differs from the original and other variations
- A step-by-step implementation plan

Return ONLY a valid Python dictionary in exactly this format, no explanation outside the dict:
{{
  "notebook_name": {{
    "methodology": "one-sentence methodology summary",
    "plan": "step-by-step plan as a single string"
  }},
  ...
}}"""}])
    log(f"=== [generate_n_variations] Step 0: Variation plans for '{original_name}' ===\n{raw_plans}\n")
    variation_plans = ast.literal_eval(raw_plans)

    paths = []
    for name, spec in variation_plans.items():
        methodology = spec["methodology"]
        plan = spec["plan"]
        log(f"--- [generate_n_variations] Generating variation: '{name}' ---")
        log(f"Methodology: {methodology}\n")

        raw = _call([{"role": "user", "content": f"""You are an expert ML engineer. Generate a Jupyter notebook as valid JSON implementing this methodologically distinct variation of an existing notebook.

Original notebook:
{original_content}

Pipeline methodology: {methodology}

Implementation plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- Implement exactly the pipeline methodology described above — do not fall back to the original approach
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}])
        log(f"=== [generate_n_variations] Raw notebook JSON for '{name}' (first 300 chars) ===\n{raw[:300]}\n")

        nb = _parse_notebook_raw(raw)
        path = os.path.join(variations_dir, f"{name}.ipynb")
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        meta_path = os.path.join(variations_dir, f"{name}.json")
        with open(meta_path, "w") as f:
            json.dump({"methodology": methodology, "plan": plan}, f, indent=2)
        log(f"Variation saved to {path}")
        paths.append(path)

    return paths

In [25]:
def generate_n_subvariations(notebook_path: str, n: int = 3, overwrite: bool = False) -> list:
    # Resolve parent metadata from sidecar JSON saved by generate_n_variations
    variations_dir = os.path.dirname(notebook_path)
    parent_name = os.path.splitext(os.path.basename(notebook_path))[0]
    meta_path = os.path.join(variations_dir, f"{parent_name}.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    parent_methodology = meta["methodology"]
    parent_plan = meta["plan"]

    # Resolve original (root) notebook content
    original_dir_name = os.path.basename(variations_dir)
    if original_dir_name.startswith("variations_"):
        original_base = original_dir_name[len("variations_"):]
    else:
        original_base = original_dir_name
    notebook_dir = os.path.dirname(variations_dir)
    original_notebook_path = os.path.join(notebook_dir, original_base + ".ipynb")
    with open(original_notebook_path, "r") as f:
        original_nb = json.load(f)
    cells_text = []
    for cell in original_nb.get("cells", []):
        source = "".join(cell.get("source", []))
        if source.strip():
            cells_text.append(source)
    original_content = "\n\n---\n\n".join(cells_text)

    subvariations_dir = os.path.join(variations_dir, f"variations_{parent_name}")
    if os.path.exists(subvariations_dir) and not overwrite:
        return []
    os.makedirs(subvariations_dir, exist_ok=True)

    log_path = os.path.join(subvariations_dir, "generation.log")

    def log(msg: str):
        with open(log_path, "a") as lf:
            lf.write(msg + "\n")

    # Step 0: Plan all subvariations upfront
    raw_subplans = _call([{"role": "user", "content": f"""You are an expert ML engineer.

Your task is to generate {n} SUBVARIATIONS of an existing notebook variation.

Original notebook:
{original_content}

Parent variation name:
{parent_name}

Parent variation methodology:
{parent_methodology}

Parent variation implementation plan:
{parent_plan}

Goal:
Generate subvariations that stay within the SAME methodology family as the parent variation.
These are not new top-level variations. They must be controlled descendants of the parent method.

Requirements for every subvariation:
{REQUIREMENTS}
- Stay on the same task type as the original notebook
- Preserve the parent variation's overall inference paradigm
- Do NOT switch to a different model family or task framing unless that change is a minor internal variant of the same method
- Each subvariation should change only 1-2 meaningful internal design choices
- The set of subvariations should be diverse with respect to internal knobs, but all must remain recognizably the same parent methodology
- Prefer meaningful internal axes such as thresholding, pooling, prompting, verbalizer choice, aggregation, normalization, decoding, or prototype construction
- Avoid cosmetic changes
- Avoid producing duplicates of the parent or of each other

For each subvariation, write:
- A snake_case filename (without .ipynb)
- A one-sentence methodology summary that explicitly states:
  1. what is preserved from the parent variation
  2. what internal methodological choice is changed
  3. how it differs from the sibling subvariations
- A step-by-step implementation plan

Return ONLY a valid Python dictionary in exactly this format:
{{
  "subvariation_name": {{
    "methodology": "one-sentence methodology summary",
    "plan": "step-by-step plan as a single string"
  }},
  ...
}}"""}])
    log(f"=== [generate_n_subvariations] Step 0: Subvariation plans for '{parent_name}' ===\n{raw_subplans}\n")
    subvariation_plans = ast.literal_eval(raw_subplans)

    paths = []
    for name, spec in subvariation_plans.items():
        methodology = spec["methodology"]
        plan = spec["plan"]
        log(f"--- [generate_n_subvariations] Generating subvariation: '{name}' ---")
        log(f"Methodology: {methodology}\n")

        raw = _call([{"role": "user", "content": f"""You are an expert ML engineer. Generate a Jupyter notebook as valid JSON implementing this subvariation of a parent variation notebook.

Original notebook:
{original_content}

Parent variation methodology: {parent_methodology}

Subvariation methodology: {methodology}

Implementation plan:
{plan}

Requirements:
{REQUIREMENTS}
- Use HuggingFace datasets to load data
- Include cells for imports, data loading, inference, and evaluation
- Implement exactly the subvariation methodology described above — preserve the parent paradigm, change only the specified internal design choices
- The notebook must be valid .ipynb JSON (nbformat 4)
- Return ONLY the raw JSON, no markdown fences or explanation."""}])
        log(f"=== [generate_n_subvariations] Raw notebook JSON for '{name}' (first 300 chars) ===\n{raw[:300]}\n")

        nb = _parse_notebook_raw(raw)
        path = os.path.join(subvariations_dir, f"{name}.ipynb")
        with open(path, "w") as f:
            json.dump(nb, f, indent=1)
        log(f"Subvariation saved to {path}")
        paths.append(path)

    return paths

# Dataset 1

- Select best suited task from list
- Create initial notebook and store in "../notebooks/batch_2" folder.
- Create 5 variations of the initial notebook.
- For each variation, create 2-4 new variations of the variation.

In [26]:
task_1 = "Semantic textual similarity scoring on the STS-B validation set using the sentence-transformers/all-MiniLM-L6-v2 embedding model and cosine similarity correlation evaluation on device='mps'."

In [27]:
# import shutil

# notebook_dir_1 = "../notebooks/batch_3"
# if os.path.exists(notebook_dir_1):
#     shutil.rmtree(notebook_dir_1)
# os.makedirs(notebook_dir_1)

# notebook_path_1 = generate_notebook(task_1, notebook_dir_1)

In [28]:
# variation_paths_1 = generate_n_variations(notebook_path_1, n=7)

In [29]:
# for vp in variation_paths_1:
    # generate_n_subvariations(vp, n=5)

# Dataset 2

In [30]:
task_2 = "Paraphrase detection on the GLUE MRPC dataset using a pretrained sentence-pair classification model such as DistilBERT, evaluated without any training."

In [31]:
import shutil

notebook_dir_2 = "../notebooks/batch_4"
if os.path.exists(notebook_dir_2):
    shutil.rmtree(notebook_dir_2)
os.makedirs(notebook_dir_2)

notebook_path_2 = generate_notebook(task_2, notebook_dir_2)

In [32]:
variation_paths_2 = generate_n_variations(notebook_path_2, n=7)

In [33]:
for vp in variation_paths_2:
    generate_n_subvariations(vp, n=5)